# Compare Ollama models on your codebase

This notebook packs your repository's source files into a prompt, asks every model in `MODELS` the same questions, and lays the answers and speeds side by side.

**Before running:**
1. Make sure Ollama is running (the menu bar app, or `ollama serve`).
2. Pull the models you want to compare, e.g. `ollama pull qwen2.5-coder:7b`.
3. Install the notebook dependencies in your repo:
   - with uv: `uv add --dev jupyter ollama pandas` then `uv run jupyter lab`
   - with pip: `pip install jupyter ollama pandas`

Everything runs locally; nothing leaves your Mac (as long as you don't list a `:cloud` model).

## 1. Configuration
Edit this cell, then run the notebook top to bottom.

In [ ]:
from pathlib import Path

# Models to compare (must already be pulled; see `ollama list`)
MODELS = [
    "qwen2.5-coder:7b",
    "llama3.2",
]

# Root of the repo to analyze. Default: the folder this notebook lives in.
REPO_ROOT = Path.cwd()

# Which files to include
INCLUDE_EXTENSIONS = {
    ".py", ".js", ".ts", ".tsx", ".jsx", ".go", ".rs", ".java", ".kt", ".swift",
    ".c", ".h", ".cpp", ".hpp", ".cs", ".rb", ".php", ".sql", ".sh",
    ".md", ".toml", ".yaml", ".yml", ".json",
}
EXCLUDE_DIRS = {
    ".git", ".venv", "venv", "node_modules", "__pycache__", "dist", "build",
    ".ipynb_checkpoints", ".mypy_cache", ".pytest_cache", "target", ".next",
}
EXCLUDE_FILES = {"package-lock.json", "uv.lock", "poetry.lock", "yarn.lock", "pnpm-lock.yaml"}
MAX_FILE_CHARS = 20_000          # skip individual files larger than this

# Context window to request from Ollama (tokens). Ollama's default is small and it
# silently truncates longer prompts, so this matters. Bigger = more memory.
NUM_CTX = 32_768

TEMPERATURE = 0.2                # low = more consistent answers between runs

## 2. Check that Ollama is running and the models are installed

In [ ]:
import ollama

try:
    installed = {m.model for m in ollama.list().models}
except Exception as e:
    raise RuntimeError("Can't reach Ollama at localhost:11434. Is the app open or `ollama serve` running?") from e

def is_installed(name):
    # "llama3.2" is stored as "llama3.2:latest"
    return name in installed or f"{name}:latest" in installed

missing = [m for m in MODELS if not is_installed(m)]
print("Installed models:", ", ".join(sorted(installed)) or "(none)")
if missing:
    print("\n⚠️  Not installed, will be skipped:", ", ".join(missing))
    print("   Pull them with: " + " && ".join(f"ollama pull {m}" for m in missing))
MODELS_TO_RUN = [m for m in MODELS if m not in missing]
print("\nWill compare:", MODELS_TO_RUN)

## 3. Collect the codebase
Uses `git ls-files` when the folder is a git repo (so `.gitignore` is respected), and walks the folder otherwise.

In [ ]:
import subprocess

def list_repo_files(root: Path) -> list[Path]:
    try:
        out = subprocess.run(
            ["git", "ls-files", "--cached", "--others", "--exclude-standard"],
            cwd=root, capture_output=True, text=True, check=True,
        ).stdout
        files = [root / line for line in out.splitlines() if line]
    except (subprocess.CalledProcessError, FileNotFoundError):
        files = [p for p in root.rglob("*") if p.is_file()]
    keep = []
    for p in files:
        rel = p.relative_to(root)
        if any(part in EXCLUDE_DIRS for part in rel.parts):
            continue
        if p.name in EXCLUDE_FILES or p.suffix.lower() not in INCLUDE_EXTENSIONS:
            continue
        if p.suffix == ".ipynb" or not p.is_file():
            continue
        keep.append(p)
    return sorted(keep)

def build_context(root: Path) -> tuple[str, list[str], list[str]]:
    parts, included, skipped = [], [], []
    for p in list_repo_files(root):
        rel = str(p.relative_to(root))
        try:
            text = p.read_text(encoding="utf-8")
        except (UnicodeDecodeError, OSError):
            skipped.append(f"{rel} (unreadable)")
            continue
        if len(text) > MAX_FILE_CHARS:
            skipped.append(f"{rel} ({len(text):,} chars, too large)")
            continue
        parts.append(f"===== FILE: {rel} =====\n{text}")
        included.append(rel)
    return "\n\n".join(parts), included, skipped

CODEBASE, included_files, skipped_files = build_context(REPO_ROOT)
approx_tokens = len(CODEBASE) // 4   # rough rule of thumb: ~4 characters per token

print(f"Included {len(included_files)} files, ~{approx_tokens:,} tokens")
for f in included_files:
    print("  ", f)
if skipped_files:
    print(f"\nSkipped {len(skipped_files)}:")
    for f in skipped_files:
        print("  ", f)
if approx_tokens > NUM_CTX * 0.8:
    print(f"\n⚠️  The codebase (~{approx_tokens:,} tokens) is close to or over NUM_CTX ({NUM_CTX:,}).")
    print("   Raise NUM_CTX (if you have the memory), narrow INCLUDE_EXTENSIONS, or point REPO_ROOT at a subfolder.")

## 4. Questions to ask every model
Mix broad questions with ones you already know the answer to, so you can grade the models.

In [ ]:
SYSTEM_PROMPT = (
    "You are an expert software engineer reviewing a codebase. "
    "Answer using only the code provided. Reference specific file names. "
    "If the code doesn't contain the answer, say so instead of guessing."
)

QUESTIONS = [
    "Give a high-level overview of what this codebase does and how it is organized.",
    "What are the main entry points, and what happens when the program runs?",
    "Identify up to three potential bugs or risky pieces of code, with file names.",
    # Add questions specific to your repo, e.g.:
    # "Where is configuration loaded, and which settings are supported?",
]

## 5. Run the comparison
Each model is loaded once and then unloaded, so only one occupies memory at a time. The first question per model includes load time.

In [ ]:
import time

def ask(model: str, question: str) -> dict:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Here is the codebase:\n\n{CODEBASE}\n\nQuestion: {question}"},
    ]
    start = time.perf_counter()
    r = ollama.chat(
        model=model,
        messages=messages,
        options={"num_ctx": NUM_CTX, "temperature": TEMPERATURE},
    )
    wall = time.perf_counter() - start
    gen_s = (r.eval_duration or 0) / 1e9
    return {
        "model": model,
        "question": question,
        "answer": r.message.content,
        "prompt_tokens": r.prompt_eval_count,
        "output_tokens": r.eval_count,
        "tokens_per_sec": round(r.eval_count / gen_s, 1) if gen_s else None,
        "load_s": round((r.load_duration or 0) / 1e9, 1),
        "total_s": round(wall, 1),
    }

results = []
for model in MODELS_TO_RUN:
    print(f"▶ {model}")
    for i, q in enumerate(QUESTIONS, 1):
        try:
            res = ask(model, q)
            print(f"   Q{i}: {res['total_s']}s, {res['tokens_per_sec']} tok/s")
        except Exception as e:
            res = {"model": model, "question": q, "answer": f"ERROR: {type(e).__name__}: {e}"}
            print(f"   Q{i}: error: {type(e).__name__}: {e}")
        results.append(res)
    try:
        ollama.generate(model=model, prompt="", keep_alive=0)   # unload to free memory
    except Exception:
        pass

# If prompt_tokens is far below the estimate above, the prompt was truncated: raise NUM_CTX.

## 6. Speed summary

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
if "tokens_per_sec" in df:
    summary = (
        df.groupby("model")[["tokens_per_sec", "total_s", "output_tokens", "prompt_tokens"]]
        .mean(numeric_only=True).round(1)
        .sort_values("tokens_per_sec", ascending=False)
    )
    display(summary)
else:
    print("No successful runs to summarize.")

## 7. Answers side by side

In [ ]:
from IPython.display import Markdown, display

for i, q in enumerate(QUESTIONS, 1):
    display(Markdown(f"---\n## Q{i}: {q}"))
    for r in (r for r in results if r["question"] == q):
        stats = f"{r.get('total_s', '?')}s · {r.get('tokens_per_sec', '?')} tok/s"
        display(Markdown(f"### {r['model']}  \n*{stats}*\n\n{r['answer']}"))

## 8. Save results
Writes a timestamped Markdown report next to the notebook so you can compare runs over time.

In [ ]:
from datetime import datetime

stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
out_dir = Path("model_comparisons"); out_dir.mkdir(exist_ok=True)
report = out_dir / f"comparison-{stamp}.md"

lines = [f"# Model comparison {stamp}", "", f"Repo: `{REPO_ROOT}`  ", f"Files: {len(included_files)}, ~{approx_tokens:,} tokens", ""]
for i, q in enumerate(QUESTIONS, 1):
    lines += [f"## Q{i}: {q}", ""]
    for r in (r for r in results if r["question"] == q):
        lines += [f"### {r['model']} ({r.get('total_s', '?')}s, {r.get('tokens_per_sec', '?')} tok/s)", "", r["answer"], ""]
report.write_text("\n".join(lines), encoding="utf-8")
print("Saved", report)
# Tip: add model_comparisons/ to .gitignore if you don't want to commit these.

## 9. Ask a one-off question
Handy for follow-ups to a single model without rerunning everything.

In [ ]:
def quick_ask(model: str, question: str):
    r = ask(model, question)
    display(Markdown(f"**{model}** · {r['total_s']}s · {r['tokens_per_sec']} tok/s\n\n{r['answer']}"))

# quick_ask("qwen2.5-coder:7b", "Which function would I change to add a new CLI flag?")